In [ ]:
import pandas as pd
import numpy as np



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
from sentence_transformers import SentenceTransformer

from tqdm import tqdm
import torch



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/capstone/financialData.csv", encoding='latin-1')
df

,neutral,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing ."
0,neutral,Technopolis plans to develop in stages an area...
1,negative,The international electronic industry company ...
2,positive,With the new production plant the company woul...
3,positive,According to the company 's updated strategy f...
4,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...
...,...,...
4840,negative,LONDON MarketWatch -- Share prices ended lower...
4841,neutral,Rinkuskiai 's beer sales fell by 6.5 per cent ...
4842,negative,Operating profit fell to EUR 35.4 mn from EUR ...
4843,negative,Net sales of the Paper segment decreased to EU...


In [ ]:
df = df.rename(columns={'neutral': 'Sentiment'})
df = df.rename(columns={'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .': 'Sentence'})

In [ ]:
df

,Sentiment,Sentence
0,neutral,Technopolis plans to develop in stages an area...
1,negative,The international electronic industry company ...
2,positive,With the new production plant the company woul...
3,positive,According to the company 's updated strategy f...
4,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...
...,...,...
4840,negative,LONDON MarketWatch -- Share prices ended lower...
4841,neutral,Rinkuskiai 's beer sales fell by 6.5 per cent ...
4842,negative,Operating profit fell to EUR 35.4 mn from EUR ...
4843,negative,Net sales of the Paper segment decreased to EU...


In [ ]:
df['Sentiment'].value_counts()

,count
Sentiment,
neutral,2878
positive,1363
negative,604


In [ ]:
df.isnull().sum()

,0
Sentiment,0
Sentence,0


In [ ]:
X = df['Sentence']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Text Vectorization with TF-IDF biagrams for logistic regression

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,      # limit vocab size (tune as needed)
    ngram_range=(1,2),      # unigrams + bigrams
    stop_words='english'
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
clf = LogisticRegression(max_iter=1000, solver='lbfgs')
clf.fit(X_train_vec, y_train)


LogisticRegression(max_iter=1000)

In [ ]:
y_pred = clf.predict(X_test_vec)

print("\nModel Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Model Performance:
Accuracy: 0.7440660474716202

Classification Report:
              precision    recall  f1-score   support

    negative       0.72      0.39      0.51       121
     neutral       0.75      0.94      0.83       576
    positive       0.73      0.48      0.58       272

    accuracy                           0.74       969
   macro avg       0.73      0.60      0.64       969
weighted avg       0.74      0.74      0.72       969



In [ ]:
test_sentences = [
   "TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars."
]

test_vec = vectorizer.transform(test_sentences)
preds = clf.predict(test_vec)

print("\nCustom Sentence Predictions:")
for sent, pred in zip(test_sentences, preds):
    print(f"{sent} --> {pred}")


Custom Sentence Predictions:
TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars. --> neutral


Pretrained FinBERT

In [ ]:
# Load FinBERT (domain-specific)
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# Test sentences
finance_sentences = [
     "TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars."
]

print("\nFinBERT Predictions:")
for sent in finance_sentences:
    result = finbert_pipeline(sent)[0]
    print(f"{sent} --> {result['label']} (score: {result['score']:.4f})")


FinBERT Predictions:
TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars. --> negative (score: 0.8796)


In [ ]:
predictions = []
for text in tqdm(df['Sentence'], desc="Running FinBERT predictions"):
    result = finbert_pipeline(text)[0]
    predictions.append(result['label'].lower())  # to match dataset labels


Running FinBERT predictions: 100%|██████████| 4845/4845 [13:44<00:00,  5.88it/s]


In [ ]:
y_true = df['Sentiment'].str.lower()   # Ensure same casing
y_pred = predictions

print("\nFinBERT Performance on FinancialPhraseBank:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))


FinBERT Performance on FinancialPhraseBank:
Accuracy: 0.8893704850361197

Classification Report:
              precision    recall  f1-score   support

    negative     0.8016    0.9702    0.8779       604
     neutral     0.9622    0.8575    0.9069      2878
    positive     0.8102    0.9208    0.8620      1363

    accuracy                         0.8894      4845
   macro avg     0.8580    0.9162    0.8822      4845
weighted avg     0.8994    0.8894    0.8906      4845



Using embeddings with Logistic Regression

In [ ]:
X = df['Sentence'].tolist()
y = df['Sentiment'].tolist()

In [ ]:
embedder = SentenceTransformer("yiyanghkust/finbert-tone")

X_embeddings = embedder.encode(X, batch_size=32, show_progress_bar=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Batches:   0%|          | 0/152 [00:00<?, ?it/s]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42)

In [ ]:
clf = LogisticRegression(max_iter=500, class_weight="balanced")
clf.fit(X_train, y_train)

# 5. Evaluation
y_pred = clf.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8018575851393189

Classification Report:
               precision    recall  f1-score   support

    negative       0.73      0.78      0.75       115
     neutral       0.87      0.82      0.84       567
    positive       0.72      0.78      0.75       287

    accuracy                           0.80       969
   macro avg       0.77      0.79      0.78       969
weighted avg       0.81      0.80      0.80       969



In [ ]:
finance_sentences = [
     "TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars.",
     "Apple stock surges after strong earnings report",
    "Tesla shares plunge following disappointing guidance",
    "The market remains stable amid economic uncertainty"
]

In [ ]:
custom_embeddings = embedder.encode(finance_sentences, batch_size=16, show_progress_bar=True)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
predictions = clf.predict(custom_embeddings)

# 5️⃣ Display results
for headline, pred in zip(finance_sentences, predictions):
    print(f"{headline} --> {pred}")

TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars. --> neutral
Apple stock surges after strong earnings report --> positive
Tesla shares plunge following disappointing guidance --> negative
The market remains stable amid economic uncertainty --> positive


In [ ]:
import joblib

joblib.dump(clf, "logreg_sentiment2.pkl")


['logreg_sentiment2.pkl']